# RAG-система для Конституции Российской Федерации

Данный ноутбук реализует полноценную RAG-систему (Retrieval-Augmented Generation) для работы с базой знаний на основе текста Конституции Российской Федерации.

## Технологический стек

| Компонент | Технология |
|---|---|
| Эмбеддинги | `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2` |
| LLM | Ollama (`mistral` / `llama3`) или OpenAI-совместимый API |
| Векторная БД | ChromaDB |
| Фреймворк | LangChain |
| API | FastAPI |
| Реранкер | `cross-encoder/ms-marco-MiniLM-L-6-v2` |
| Web-интерфейс | Streamlit |

---
## Этап 0: Установка зависимостей

In [ ]:
# Установка всех необходимых библиотек
!pip install -q \
    langchain \
    langchain-community \
    langchain-chroma \
    chromadb \
    sentence-transformers \
    python-docx \
    fastapi \
    uvicorn[standard] \
    streamlit \
    requests \
    pydantic \
    nest-asyncio \
    httpx

print("✅ Все зависимости установлены")

---
## Этап 1: Импорты и конфигурация

In [ ]:
import os
import re
import json
import requests
from pathlib import Path
from typing import List, Dict, Optional, Tuple

# Работа с документами
from docx import Document as DocxDocument

# LangChain
from langchain.schema import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain.prompts import PromptTemplate
from langchain.chains import RetrievalQA
from langchain_community.llms import Ollama

# Реранкер
from sentence_transformers import CrossEncoder

# FastAPI
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import uvicorn
import nest_asyncio

print("✅ Импорты выполнены успешно")

In [ ]:
# ============================================================
#  КОНФИГУРАЦИЯ — отредактируйте под свою среду
# ============================================================

CONFIG = {
    # Путь к файлу Конституции (.docx)
    "DOCX_PATH": "constitutionrf.docx",

    # Модель эмбеддингов (мультиязычная, поддерживает русский)
    "EMBED_MODEL": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",

    # Директория для хранения ChromaDB
    "CHROMA_DIR": "./chroma_constitution",

    # Название коллекции в ChromaDB
    "CHROMA_COLLECTION": "constitution_rf",

    # LLM — выберите один из вариантов:
    # "ollama"  — локальный Ollama (нужно установить: https://ollama.ai)
    # "openai"  — OpenAI-совместимый API (например, LM Studio, vLLM)
    "LLM_BACKEND": "ollama",

    # Название модели для Ollama (предварительно: ollama pull mistral)
    "OLLAMA_MODEL": "mistral",
    "OLLAMA_URL": "http://localhost:11434",

    # Настройки OpenAI-совместимого API (если LLM_BACKEND == "openai")
    "OPENAI_API_BASE": "http://localhost:1234/v1",  # например, LM Studio
    "OPENAI_API_KEY": "not-needed",
    "OPENAI_MODEL": "local-model",

    # Реранкер
    "RERANKER_MODEL": "cross-encoder/ms-marco-MiniLM-L-6-v2",

    # Количество документов для первичного retrieval
    "RETRIEVAL_K": 10,

    # Количество документов после реранкинга
    "RERANK_TOP_K": 4,

    # Параметры чанкинга
    "CHUNK_SIZE": 500,
    "CHUNK_OVERLAP": 100,
}

print("✅ Конфигурация загружена")
print(json.dumps(CONFIG, ensure_ascii=False, indent=2))

---
## Этап 2: Работа с базой знаний

### 2.1 Загрузка и анализ структуры документа

In [ ]:
def load_constitution_docx(path: str) -> List[Dict]:
    """
    Загружает и парсит .docx файл Конституции РФ.
    Возвращает список параграфов с метаинформацией.
    """
    doc = DocxDocument(path)
    paragraphs = []

    current_chapter = None
    current_article = None
    chapter_num = 0
    article_num = 0

    # Регулярные выражения для определения структурных элементов
    re_chapter = re.compile(r'^Глава\s+(\d+)[\s\.]', re.IGNORECASE)
    re_article = re.compile(r'^Статья\s+(\d+)[\s\.]', re.IGNORECASE)
    re_preambule = re.compile(r'^Преамбул', re.IGNORECASE)

    for i, para in enumerate(doc.paragraphs):
        text = para.text.strip()
        if not text:
            continue

        # Определяем стиль абзаца (заголовок или обычный текст)
        style_name = para.style.name if para.style else ""

        # Проверяем принадлежность к главе
        m_chapter = re_chapter.match(text)
        if m_chapter:
            chapter_num = int(m_chapter.group(1))
            current_chapter = text
            current_article = None
            article_num = 0
            continue

        # Проверяем принадлежность к статье
        m_article = re_article.match(text)
        if m_article:
            article_num = int(m_article.group(1))
            current_article = text
            continue

        # Преамбула
        if re_preambule.match(text):
            current_chapter = "Преамбула"
            current_article = "Преамбула"
            chapter_num = 0
            article_num = 0

        # Сохраняем содержательный параграф с метаданными
        paragraphs.append({
            "text": text,
            "chapter_num": chapter_num,
            "chapter_title": current_chapter,
            "article_num": article_num,
            "article_title": current_article,
            "para_index": i,
            "style": style_name,
        })

    return paragraphs


# ─── Загрузка ───
paragraphs = load_constitution_docx(CONFIG["DOCX_PATH"])

print(f"📄 Всего параграфов: {len(paragraphs)}")
print(f"\n📌 Пример первых 5 параграфов:")
for p in paragraphs[:5]:
    print(f"  [Гл.{p['chapter_num']} | Ст.{p['article_num']}] {p['text'][:80]}...")

In [ ]:
# Анализ структуры документа
from collections import Counter

chapters = sorted(set((p['chapter_num'], p['chapter_title']) for p in paragraphs if p['chapter_title']))
articles = sorted(set(p['article_num'] for p in paragraphs if p['article_num'] > 0))

print("📚 СТРУКТУРА ДОКУМЕНТА")
print("=" * 50)
print(f"Количество глав: {len(chapters)}")
print(f"Количество статей: {len(articles)}")
print(f"Диапазон статей: {min(articles)} – {max(articles)}")
print()
print("Главы:")
for num, title in chapters:
    art_in_chapter = [p for p in paragraphs if p['chapter_num'] == num]
    arts = sorted(set(p['article_num'] for p in art_in_chapter if p['article_num'] > 0))
    print(f"  Глава {num}: {title[:60]} | Статей: {len(arts)}")

### 2.2 Чанкирование документа

In [ ]:
def create_chunks(paragraphs: List[Dict]) -> List[Document]:
    """
    Создаёт чанки из параграфов документа.
    
    Стратегия:
    - Параграфы одной статьи объединяются в блок
    - Блок разбивается RecursiveCharacterTextSplitter
    - Метаинформация (глава, статья) сохраняется в metadata
    """
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CONFIG["CHUNK_SIZE"],
        chunk_overlap=CONFIG["CHUNK_OVERLAP"],
        separators=["\n\n", "\n", ". ", " ", ""],
    )

    documents: List[Document] = []

    # Группируем параграфы по статьям
    articles_map: Dict[int, Dict] = {}
    for p in paragraphs:
        art_num = p['article_num']
        if art_num not in articles_map:
            articles_map[art_num] = {
                "texts": [],
                "chapter_num": p['chapter_num'],
                "chapter_title": p['chapter_title'] or "",
                "article_title": p['article_title'] or "",
                "article_num": art_num,
            }
        articles_map[art_num]["texts"].append(p['text'])

    # Для каждой статьи создаём чанки
    for art_num, art_data in sorted(articles_map.items()):
        full_text = "\n".join(art_data["texts"])

        # Префикс с заголовком статьи для контекста
        if art_num > 0:
            prefix = f"Статья {art_num}.\n"
        else:
            prefix = "Преамбула.\n"

        chunks = splitter.split_text(prefix + full_text)

        for chunk_idx, chunk_text in enumerate(chunks):
            metadata = {
                "article_num": art_num,
                "article_title": art_data["article_title"],
                "chapter_num": art_data["chapter_num"],
                "chapter_title": art_data["chapter_title"],
                "chunk_index": chunk_idx,
                "source": f"Конституция РФ, Ст.{art_num}, ч.{chunk_idx+1}",
            }
            documents.append(Document(page_content=chunk_text, metadata=metadata))

    return documents


chunks = create_chunks(paragraphs)

print(f"✂️  Всего чанков: {len(chunks)}")
print(f"\n📌 Пример чанков:")
for doc in chunks[:3]:
    print(f"\n{'─'*60}")
    print(f"  Источник: {doc.metadata['source']}")
    print(f"  Глава {doc.metadata['chapter_num']}: {doc.metadata['chapter_title'][:50]}")
    print(f"  Текст: {doc.page_content[:200]}...")

In [ ]:
# Статистика по чанкам
lengths = [len(c.page_content) for c in chunks]
import statistics

print("📊 СТАТИСТИКА ЧАНКОВ")
print(f"  Минимальный размер: {min(lengths)} символов")
print(f"  Максимальный размер: {max(lengths)} символов")
print(f"  Средний размер:     {statistics.mean(lengths):.0f} символов")
print(f"  Медиана:            {statistics.median(lengths):.0f} символов")

### 2.3–2.4 Векторизация и сохранение в ChromaDB

In [ ]:
print("⏳ Загрузка модели эмбеддингов...")
embeddings = HuggingFaceEmbeddings(
    model_name=CONFIG["EMBED_MODEL"],
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)
print(f"✅ Модель '{CONFIG['EMBED_MODEL']}' загружена")

# Тестовый эмбеддинг
test_emb = embeddings.embed_query("права человека")
print(f"   Размерность вектора: {len(test_emb)}")

In [ ]:
import shutil

# Удаляем старую коллекцию (если есть), для чистого старта
if Path(CONFIG["CHROMA_DIR"]).exists():
    shutil.rmtree(CONFIG["CHROMA_DIR"])
    print("🗑️  Старая коллекция удалена")

print("⏳ Векторизация и сохранение в ChromaDB (может занять 1–5 минут)...")

# Создаём векторное хранилище
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=CONFIG["CHROMA_DIR"],
    collection_name=CONFIG["CHROMA_COLLECTION"],
)

print(f"✅ Векторная база создана: {CONFIG['CHROMA_DIR']}")
print(f"   Документов в хранилище: {vectorstore._collection.count()}")

In [ ]:
# Проверка — поиск без реранкера
test_query = "право на образование"
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
results = retriever.invoke(test_query)

print(f"🔍 Тестовый поиск: '{test_query}'")
for i, doc in enumerate(results, 1):
    print(f"\n  [{i}] {doc.metadata['source']}")
    print(f"      {doc.page_content[:150]}...")

---
## Этап 3: Реранкер и промпт-инжиниринг

In [ ]:
# ─── 3.2 Загрузка реранкера ───
print(f"⏳ Загрузка реранкера '{CONFIG['RERANKER_MODEL']}'...")
reranker = CrossEncoder(CONFIG["RERANKER_MODEL"])
print("✅ Реранкер загружен")


def rerank_documents(
    query: str,
    docs: List[Document],
    top_k: int = CONFIG["RERANK_TOP_K"]
) -> List[Document]:
    """
    Переранжирует список документов с помощью Cross-Encoder реранкера.
    Возвращает top_k наиболее релевантных документов.
    """
    if not docs:
        return []

    # Формируем пары (запрос, документ) для реранкера
    pairs = [(query, doc.page_content) for doc in docs]
    scores = reranker.predict(pairs)

    # Сортируем по убыванию релевантности
    scored_docs = sorted(
        zip(scores, docs),
        key=lambda x: x[0],
        reverse=True
    )

    top_docs = [doc for _, doc in scored_docs[:top_k]]
    top_scores = [float(score) for score, _ in scored_docs[:top_k]]

    print(f"  📊 Реранкинг: отобрано {len(top_docs)} из {len(docs)} документов")
    for score, doc in zip(top_scores, top_docs):
        print(f"     score={score:.4f} | {doc.metadata['source']}")

    return top_docs


# Тест реранкера
test_docs = retriever.invoke(test_query)
print(f"\n🔁 Применяем реранкер для: '{test_query}'")
reranked = rerank_documents(test_query, test_docs)

In [ ]:
# ─── 3.3 Системный промпт ───

SYSTEM_PROMPT_TEMPLATE = """\
Ты — юридический ассистент, специализирующийся на Конституции Российской Федерации.
Твоя задача — точно и грамотно отвечать на вопросы пользователей, опираясь 
исключительно на предоставленный контекст из Конституции РФ.

== КОНТЕКСТ ИЗ КОНСТИТУЦИИ РФ ==
{context}

== ИНСТРУКЦИИ ДЛЯ ОТВЕТА ==
1. Отвечай ТОЛЬКО на основании предоставленного контекста. Не придумывай информацию.
2. Если ответ на вопрос отсутствует в контексте, прямо сообщи об этом.
3. Обязательно указывай номера статей, из которых взята информация.
4. Структурируй ответ: сначала краткий вывод, затем цитаты из статей.
5. Используй профессиональный юридический язык.
6. Не допускай галлюцинаций — каждое утверждение должно подкрепляться контекстом.
7. Если вопрос неоднозначен, разбери все возможные трактовки по тексту Конституции.

== ВОПРОС ПОЛЬЗОВАТЕЛЯ ==
{question}

== ОТВЕТ ==
"""

PROMPT = PromptTemplate(
    input_variables=["context", "question"],
    template=SYSTEM_PROMPT_TEMPLATE,
)

print("✅ Системный промпт сформирован")
print("\n📝 Шаблон промпта:")
print(SYSTEM_PROMPT_TEMPLATE[:400] + "...")

In [ ]:
# ─── 3.1 Инициализация LLM ───

def get_llm():
    """Возвращает LLM в зависимости от выбранного бэкенда."""
    if CONFIG["LLM_BACKEND"] == "ollama":
        from langchain_community.llms import Ollama
        llm = Ollama(
            model=CONFIG["OLLAMA_MODEL"],
            base_url=CONFIG["OLLAMA_URL"],
            temperature=0.1,
        )
        print(f"✅ LLM: Ollama ({CONFIG['OLLAMA_MODEL']}) @ {CONFIG['OLLAMA_URL']}")
    elif CONFIG["LLM_BACKEND"] == "openai":
        from langchain_community.llms import OpenAI
        llm = OpenAI(
            model_name=CONFIG["OPENAI_MODEL"],
            openai_api_base=CONFIG["OPENAI_API_BASE"],
            openai_api_key=CONFIG["OPENAI_API_KEY"],
            temperature=0.1,
        )
        print(f"✅ LLM: OpenAI-compatible API @ {CONFIG['OPENAI_API_BASE']}")
    else:
        raise ValueError(f"Неизвестный LLM_BACKEND: {CONFIG['LLM_BACKEND']}")
    return llm


llm = get_llm()

### 3.4 Основная функция RAG-пайплайна

In [ ]:
def format_context(docs: List[Document]) -> str:
    """
    Форматирует список документов в читаемый контекст для промпта.
    Включает метаинформацию о каждом чанке.
    """
    parts = []
    for i, doc in enumerate(docs, 1):
        meta = doc.metadata
        header = (
            f"[Документ {i}] "
            f"Глава {meta.get('chapter_num', '?')}, "
            f"Статья {meta.get('article_num', '?')} — "
            f"{meta.get('chapter_title', '')[:40]}"
        )
        parts.append(f"{header}\n{doc.page_content}")
    return "\n\n" + "─" * 50 + "\n".join(parts)


def rag_query(
    query: str,
    verbose: bool = True
) -> Dict:
    """
    Полный RAG-пайплайн:
    1. Retrieval — поиск в ChromaDB
    2. Reranking — переранжирование Cross-Encoder
    3. Generation — генерация ответа LLM

    Возвращает словарь:
        {
            'answer': str,
            'sources': List[dict],
            'retrieved_count': int,
            'reranked_count': int,
        }
    """
    if verbose:
        print(f"\n{'='*60}")
        print(f"🔎 Запрос: {query}")
        print(f"{'='*60}")

    # ── Шаг 1: Retrieval ──
    if verbose:
        print(f"\n📥 Шаг 1: Retrieval (k={CONFIG['RETRIEVAL_K']})")
    retrieved_docs = vectorstore.similarity_search(
        query,
        k=CONFIG["RETRIEVAL_K"]
    )
    if verbose:
        print(f"   Найдено документов: {len(retrieved_docs)}")

    # ── Шаг 2: Reranking ──
    if verbose:
        print(f"\n🔁 Шаг 2: Reranking (top_k={CONFIG['RERANK_TOP_K']})")
    final_docs = rerank_documents(query, retrieved_docs, CONFIG["RERANK_TOP_K"])

    # ── Шаг 3: Формирование промпта и генерация ──
    if verbose:
        print(f"\n🤖 Шаг 3: Генерация ответа LLM")

    context = format_context(final_docs)
    prompt_text = PROMPT.format(context=context, question=query)

    answer = llm.invoke(prompt_text)

    # Формируем источники для ответа
    sources = [
        {
            "source": doc.metadata.get("source", ""),
            "article_num": doc.metadata.get("article_num", 0),
            "chapter_num": doc.metadata.get("chapter_num", 0),
            "chapter_title": doc.metadata.get("chapter_title", ""),
            "text_preview": doc.page_content[:150] + "...",
        }
        for doc in final_docs
    ]

    result = {
        "answer": answer.strip(),
        "sources": sources,
        "retrieved_count": len(retrieved_docs),
        "reranked_count": len(final_docs),
    }

    if verbose:
        print(f"\n✅ Ответ сформирован")
        print(f"{'─'*60}")
        print(answer.strip())
        print(f"\n📎 Источники ({len(sources)}):")
        for s in sources:
            print(f"   • {s['source']}")

    return result


print("✅ RAG-пайплайн готов")

In [ ]:
# ─── Тестирование RAG-системы ───
test_questions = [
    "Какие права есть у граждан России?",
    "Кто является главой государства в России?",
    "Что говорит Конституция о праве на образование?",
    "Сколько депутатов в Государственной Думе?",
]

# Запускаем первый тест
result = rag_query(test_questions[0])

In [ ]:
# Дополнительные тесты
for q in test_questions[1:]:
    result = rag_query(q)
    print("\n" + "="*60 + "\n")

---
## Этап 4: Разработка API (FastAPI)

In [ ]:
# ─── Pydantic схемы ───

class QueryRequest(BaseModel):
    """Схема входящего запроса."""
    question: str
    top_k: Optional[int] = None  # Переопределить RERANK_TOP_K


class SourceItem(BaseModel):
    """Источник из Конституции."""
    source: str
    article_num: int
    chapter_num: int
    chapter_title: str
    text_preview: str


class QueryResponse(BaseModel):
    """Схема ответа API."""
    answer: str
    sources: List[SourceItem]
    retrieved_count: int
    reranked_count: int


# ─── FastAPI приложение ───
app = FastAPI(
    title="RAG API — Конституция РФ",
    description="API для вопросно-ответной системы на основе Конституции Российской Федерации",
    version="1.0.0",
)


@app.get("/health", tags=["Служебные"])
def health_check():
    """Проверка работоспособности сервиса."""
    return {"status": "ok", "service": "RAG Constitution RF"}


@app.post("/ask", response_model=QueryResponse, tags=["RAG"])
def ask_question(request: QueryRequest):
    """
    **Основной эндпоинт RAG-системы.**

    Принимает вопрос пользователя, выполняет поиск по базе знаний
    (Конституции РФ), применяет реранкинг и возвращает ответ LLM
    с указанием источников.

    - **question**: вопрос пользователя на русском языке
    - **top_k**: (опционально) количество документов после реранкинга
    """
    if not request.question.strip():
        raise HTTPException(status_code=400, detail="Вопрос не может быть пустым")

    # Временно переопределяем top_k, если передан
    original_top_k = CONFIG["RERANK_TOP_K"]
    if request.top_k is not None:
        CONFIG["RERANK_TOP_K"] = request.top_k

    try:
        result = rag_query(request.question, verbose=False)
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Ошибка генерации: {str(e)}")
    finally:
        CONFIG["RERANK_TOP_K"] = original_top_k

    return QueryResponse(**result)


@app.get("/stats", tags=["Служебные"])
def get_stats():
    """Статистика базы знаний."""
    return {
        "total_chunks": vectorstore._collection.count(),
        "embed_model": CONFIG["EMBED_MODEL"],
        "llm_backend": CONFIG["LLM_BACKEND"],
        "reranker": CONFIG["RERANKER_MODEL"],
    }


print("✅ FastAPI приложение создано")
print("\n📋 Эндпоинты:")
print("   GET  /health  — проверка работоспособности")
print("   POST /ask     — задать вопрос")
print("   GET  /stats   — статистика БД")
print("   GET  /docs    — Swagger UI")

In [ ]:
# ─── Запуск API сервера в фоне ───
import threading

nest_asyncio.apply()

API_PORT = 8000

def run_api():
    uvicorn.run(app, host="0.0.0.0", port=API_PORT, log_level="warning")

api_thread = threading.Thread(target=run_api, daemon=True)
api_thread.start()

import time
time.sleep(2)

# Проверка что API запущен
try:
    r = requests.get(f"http://localhost:{API_PORT}/health")
    print(f"✅ API запущен: http://localhost:{API_PORT}")
    print(f"   Swagger UI:  http://localhost:{API_PORT}/docs")
    print(f"   Health:      {r.json()}")
except Exception as e:
    print(f"❌ Ошибка запуска API: {e}")

In [ ]:
# ─── Тест API через HTTP ───
test_payload = {"question": "Каков срок полномочий Президента России?"}

response = requests.post(
    f"http://localhost:{API_PORT}/ask",
    json=test_payload,
    timeout=120
)

if response.status_code == 200:
    data = response.json()
    print("✅ API ответил успешно")
    print(f"\n📝 Ответ:\n{data['answer']}")
    print(f"\n📎 Источники:")
    for s in data['sources']:
        print(f"   • {s['source']}")
else:
    print(f"❌ Ошибка API: {response.status_code} — {response.text}")

---
## Этап 5: Web-интерфейс (Streamlit)

Код для web-интерфейса сохраняется в отдельный файл `app.py` и запускается командой:
```bash
streamlit run app.py
```

In [ ]:
# ─── Генерация файла app.py для Streamlit ───

STREAMLIT_APP = '''
import streamlit as st
import requests
import json
from datetime import datetime

# ─── Конфигурация ───
API_URL = "http://localhost:8000"

st.set_page_config(
    page_title="Конституция РФ — Умный поиск",
    page_icon="⚖️",
    layout="wide",
)

# ─── CSS стили ───
st.markdown("""
<style>
    @import url(\'https://fonts.googleapis.com/css2?family=Playfair+Display:wght@600;700&family=Source+Serif+4:ital,opsz,wght@0,8..60,300;0,8..60,400;1,8..60,300&display=swap\');

    html, body, [class*="css"] {
        font-family: \'Source Serif 4\', serif;
    }
    .main-title {
        font-family: \'Playfair Display\', serif;
        font-size: 2.4rem;
        font-weight: 700;
        color: #1a1a2e;
        text-align: center;
        margin-bottom: 0.2rem;
    }
    .subtitle {
        text-align: center;
        color: #666;
        font-size: 1rem;
        margin-bottom: 2rem;
    }
    .answer-box {
        background: #f8f6f0;
        border-left: 4px solid #b5282a;
        padding: 1.2rem 1.5rem;
        border-radius: 0 8px 8px 0;
        font-size: 1.05rem;
        line-height: 1.8;
        color: #1a1a2e;
        margin: 1rem 0;
    }
    .source-tag {
        display: inline-block;
        background: #1a1a2e;
        color: #fff;
        font-size: 0.75rem;
        padding: 2px 10px;
        border-radius: 12px;
        margin: 2px;
        font-family: monospace;
    }
    .chat-user {
        background: #e8e4f0;
        border-radius: 18px 18px 4px 18px;
        padding: 0.8rem 1.2rem;
        margin: 0.5rem 0;
        max-width: 80%;
        margin-left: auto;
        font-size: 0.97rem;
    }
    .chat-bot {
        background: #f8f6f0;
        border-radius: 18px 18px 18px 4px;
        padding: 0.8rem 1.2rem;
        border-left: 3px solid #b5282a;
        margin: 0.5rem 0;
        max-width: 90%;
        font-size: 0.97rem;
        line-height: 1.7;
    }
    .stTextInput > div > div > input {
        border-radius: 24px;
        border: 2px solid #1a1a2e;
        padding: 0.7rem 1.2rem;
        font-size: 1rem;
    }
    .stButton > button {
        background-color: #b5282a;
        color: white;
        border-radius: 24px;
        border: none;
        padding: 0.6rem 2rem;
        font-size: 1rem;
        font-weight: 600;
        width: 100%;
    }
    .stButton > button:hover {
        background-color: #8b1f21;
    }
</style>
""", unsafe_allow_html=True)

# ─── Заголовок ───
st.markdown(\'<div class="main-title">⚖️ Конституция РФ</div>\', unsafe_allow_html=True)
st.markdown(\'<div class="subtitle">Интеллектуальная поисковая система на базе RAG</div>\', unsafe_allow_html=True)

# ─── История чата ───
if "history" not in st.session_state:
    st.session_state.history = []

# ─── Боковая панель ───
with st.sidebar:
    st.markdown("### ⚙️ Настройки")
    top_k = st.slider("Документов для ответа", 2, 8, 4)

    st.markdown("### 💡 Примеры вопросов")
    example_questions = [
        "Какие основные права гарантирует Конституция?",
        "Кто избирает Президента России?",
        "Что такое федеральное собрание?",
        "Право на труд в Конституции РФ",
        "Полномочия Государственной Думы",
    ]
    for eq in example_questions:
        if st.button(eq, key=f"ex_{eq[:20]}"):
            st.session_state["input_question"] = eq

    st.markdown("---")
    if st.button("🗑️ Очистить историю"):
        st.session_state.history = []
        st.rerun()

    # Статус API
    try:
        r = requests.get(f"{API_URL}/health", timeout=2)
        st.success("🟢 API подключён")
        stats = requests.get(f"{API_URL}/stats", timeout=2).json()
        st.caption(f"Чанков в БД: {stats.get(\'total_chunks\', \'?\')}")
    except:
        st.error("🔴 API недоступен")

# ─── Отображение истории чата ───
chat_container = st.container()
with chat_container:
    for msg in st.session_state.history:
        if msg["role"] == "user":
            st.markdown(f\'<div class="chat-user">🧑 {msg["content"]}</div>\', unsafe_allow_html=True)
        else:
            st.markdown(f\'<div class="chat-bot">⚖️ {msg["content"]}</div>\', unsafe_allow_html=True)
            if msg.get("sources"):
                sources_html = " ".join([
                    f\'<span class="source-tag">{s["source"]}</span>\'
                    for s in msg["sources"]
                ])
                st.markdown(f"📎 {sources_html}", unsafe_allow_html=True)

# ─── Поле ввода ───
st.markdown("---")
col1, col2 = st.columns([5, 1])

with col1:
    default_val = st.session_state.pop("input_question", "")
    question = st.text_input(
        "Ваш вопрос",
        value=default_val,
        placeholder="Например: Какие права гарантирует Конституция?",
        label_visibility="collapsed",
    )

with col2:
    send = st.button("Спросить")

# ─── Обработка запроса ───
if send and question.strip():
    st.session_state.history.append({"role": "user", "content": question})

    with st.spinner("Ищем в Конституции..."):
        try:
            resp = requests.post(
                f"{API_URL}/ask",
                json={"question": question, "top_k": top_k},
                timeout=180,
            )
            if resp.status_code == 200:
                data = resp.json()
                st.session_state.history.append({
                    "role": "assistant",
                    "content": data["answer"],
                    "sources": data["sources"],
                })
            else:
                st.error(f"Ошибка API: {resp.status_code}")
        except requests.exceptions.Timeout:
            st.error("⏱️ Превышено время ожидания. Попробуйте ещё раз.")
        except Exception as e:
            st.error(f"Ошибка: {str(e)}")

    st.rerun()
'''

with open("app.py", "w", encoding="utf-8") as f:
    f.write(STREAMLIT_APP)

print("✅ Файл app.py создан")
print("\n🚀 Для запуска web-интерфейса выполните:")
print("   streamlit run app.py")

---
## Итоговая проверка системы

Тест на соответствие системному промпту: проверяем, что модель не галлюцинирует и отвечает только на основе контекста.

In [ ]:
# ─── Тест на следование инструкциям промпта ───

compliance_tests = [
    {
        "name": "Отсутствие галлюцинаций",
        "question": "Что говорит Конституция о полётах на Луну?",
        "check": lambda ans: any(w in ans.lower() for w in
                                  ["не содержит", "отсутствует", "не упоминает", "нет информации", "не предусмотрено"]),
    },
    {
        "name": "Указание статей",
        "question": "Каков возраст для избрания Президентом?",
        "check": lambda ans: re.search(r'[Сс]татья|ст\.\s*\d+', ans) is not None,
    },
    {
        "name": "Ответ на вопрос о правах",
        "question": "Имеет ли гражданин право на медицинскую помощь?",
        "check": lambda ans: any(w in ans.lower() for w in ["право", "медицин", "охрана здоровья"]),
    },
]

print("🧪 ТЕСТИРОВАНИЕ СИСТЕМНОГО ПРОМПТА")
print("=" * 60)

all_passed = True
for test in compliance_tests:
    result = rag_query(test["question"], verbose=False)
    passed = test["check"](result["answer"])
    status = "✅ ПРОЙДЕН" if passed else "❌ НЕ ПРОЙДЕН"
    all_passed = all_passed and passed

    print(f"\n  [{status}] {test['name']}")
    print(f"  Вопрос: {test['question']}")
    print(f"  Ответ:  {result['answer'][:200]}...")

print(f"\n{'='*60}")
print(f"ИТОГ: {'✅ Все тесты пройдены' if all_passed else '⚠️ Некоторые тесты не пройдены'}")

In [ ]:
# ─── Итоговая сводка ───
print("""
╔══════════════════════════════════════════════════════════╗
║         RAG-СИСТЕМА ДЛЯ КОНСТИТУЦИИ РФ — ГОТОВА         ║
╠══════════════════════════════════════════════════════════╣
║  ✅ Этап 1: Технологический стек определён              ║
║  ✅ Этап 2: База знаний загружена и векторизирована      ║
║  ✅ Этап 3: Реранкер подключён, промпт сформирован       ║
║  ✅ Этап 4: API (FastAPI) запущен на порту 8000          ║
║  ✅ Этап 5: Streamlit-интерфейс (app.py) создан         ║
╠══════════════════════════════════════════════════════════╣
║  Swagger UI:  http://localhost:8000/docs                ║
║  Web-UI:      streamlit run app.py                      ║
╚══════════════════════════════════════════════════════════╝
""")